<a href="https://colab.research.google.com/github/ShubhangiDimri/Indic-Meme-Understanding-Sentiment-Analysis-IMUSA-/blob/main/notebooks/IMUSA_Multimodal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

CUDA: True
GPU: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import pandas as pd

IMUSA_DIR = "/content/drive/MyDrive/IMUSA"

print("Root:")
print(os.listdir(IMUSA_DIR))

print("\nText results:")
print(os.listdir(f"{IMUSA_DIR}/results/text_muril"))

print("\nImage results:")
print(os.listdir(f"{IMUSA_DIR}/results/image_clip"))

train_split = pd.read_csv(f"{IMUSA_DIR}/split_train.csv")
val_split = pd.read_csv(f"{IMUSA_DIR}/split_val.csv")

print("\nTrain:", train_split.shape)
print("Val:", val_split.shape)

Root:
['split_train.csv', 'split_val.csv', 'results', 'Training_images']

Text results:
['best_model.pt', 'training_log.csv', 'val_predictions.csv']

Image results:
['best_model.pt', 'training_log.csv', 'val_predictions.csv']

Train: (2402, 3)
Val: (600, 3)


In [4]:
!pip install -q transformers sentencepiece accelerate scikit-learn pillow

In [5]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import CLIPModel, CLIPProcessor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# LABELS
# -------------------------
labels = ["Motivational", "Neutral", "Offensive", "Sarcasm"]

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

# -------------------------
# LOAD MuRIL
# -------------------------
MURIL_NAME = "google/muril-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MURIL_NAME)

muril_model = AutoModelForSequenceClassification.from_pretrained(
    MURIL_NAME,
    num_labels=4,
    label2id=label2id,
    id2label=id2label
)

muril_ckpt = torch.load(
    f"{IMUSA_DIR}/results/text_muril/best_model.pt",
    map_location="cpu"
)

muril_model.load_state_dict(
    muril_ckpt["model_state_dict"]
)

muril_model = muril_model.to(device)

# -------------------------
# LOAD CLIP
# -------------------------
CLIP_NAME = "openai/clip-vit-base-patch32"

clip_processor = CLIPProcessor.from_pretrained(CLIP_NAME)
clip_model = CLIPModel.from_pretrained(CLIP_NAME)

# Recreate same image classifier structure used before
class CLIPImageClassifier(nn.Module):
    def __init__(self, clip_model, num_classes=4):
        super().__init__()

        self.vision_model = clip_model.vision_model

        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(768, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, pixel_values):
        outputs = self.vision_model(
            pixel_values=pixel_values
        )

        features = outputs.pooler_output
        logits = self.classifier(features)

        return logits


image_model = CLIPImageClassifier(
    clip_model,
    num_classes=4
)

clip_ckpt = torch.load(
    f"{IMUSA_DIR}/results/image_clip/best_model.pt",
    map_location="cpu"
)

image_model.load_state_dict(
    clip_ckpt["model_state_dict"]
)

image_model = image_model.to(device)

print("MuRIL loaded:", next(muril_model.parameters()).device)
print("CLIP loaded:", next(image_model.parameters()).device)
print("✅ Saved checkpoints restored.")

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  953MB            

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

MuRIL loaded: cuda:0
CLIP loaded: cuda:0
✅ Saved checkpoints restored.


In [6]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

IMAGE_DIR = f"{IMUSA_DIR}/Training_images"
MAX_LENGTH = 128

def resolve_image_path(image_id):
    image_id = str(image_id).strip()

    # Exact filename
    path = os.path.join(IMAGE_DIR, image_id)
    if os.path.exists(path):
        return path

    # Handle the 10 IDs missing extensions
    for ext in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
        path = os.path.join(IMAGE_DIR, image_id + ext)
        if os.path.exists(path):
            return path

    return None


class IMUSAMultimodalDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # ---------- TEXT ----------
        text = str(row["Text"]) if pd.notna(row["Text"]) else ""

        text_encoding = tokenizer(
            text,
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        # ---------- IMAGE ----------
        image_id = row["Id"]
        image_path = resolve_image_path(image_id)

        if image_path is None:
            raise FileNotFoundError(f"Image not found: {image_id}")

        image = Image.open(image_path).convert("RGB")

        image_encoding = clip_processor(
            images=image,
            return_tensors="pt"
        )

        # ---------- LABEL ----------
        label = label2id[row["Category"]]

        return {
            "id": image_id,

            "input_ids":
                text_encoding["input_ids"].squeeze(0),

            "attention_mask":
                text_encoding["attention_mask"].squeeze(0),

            "pixel_values":
                image_encoding["pixel_values"].squeeze(0),

            "label":
                torch.tensor(label, dtype=torch.long)
        }


train_mm_dataset = IMUSAMultimodalDataset(train_split)
val_mm_dataset = IMUSAMultimodalDataset(val_split)

print("Train:", len(train_mm_dataset))
print("Validation:", len(val_mm_dataset))

Train: 2402
Validation: 600


In [7]:
BATCH_SIZE = 8

train_mm_loader = DataLoader(
    train_mm_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_mm_loader = DataLoader(
    val_mm_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

batch = next(iter(train_mm_loader))

print("Text:", batch["input_ids"].shape)
print("Attention:", batch["attention_mask"].shape)
print("Images:", batch["pixel_values"].shape)
print("Labels:", batch["label"].shape)
print("IDs:", batch["id"][:3])

Text: torch.Size([8, 128])
Attention: torch.Size([8, 128])
Images: torch.Size([8, 3, 224, 224])
Labels: torch.Size([8])
IDs: ['image_punjabi_2099.jpg', 'image_punjabi_2425.jpg', 'image_punjabi_3405.jpg']


In [8]:
import torch
import torch.nn as nn

class MuRILCLIPFusion(nn.Module):
    def __init__(self, muril_model, image_model, num_classes=4):
        super().__init__()

        # Use the pretrained encoders
        self.text_encoder = muril_model.bert
        self.image_encoder = image_model.vision_model

        # Freeze both encoders first
        for p in self.text_encoder.parameters():
            p.requires_grad = False

        for p in self.image_encoder.parameters():
            p.requires_grad = False

        # Both outputs are 768-d
        self.text_projection = nn.Sequential(
            nn.Linear(768, 256),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.image_projection = nn.Sequential(
            nn.Linear(768, 256),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.fusion_classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 4)
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        pixel_values
    ):

        # -------- TEXT --------
        with torch.no_grad():
            text_outputs = self.text_encoder(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            # CLS token
            text_features = text_outputs.last_hidden_state[:, 0, :]

        # -------- IMAGE --------
        with torch.no_grad():
            image_outputs = self.image_encoder(
                pixel_values=pixel_values
            )

            image_features = image_outputs.pooler_output

        # -------- PROJECT --------
        text_features = self.text_projection(text_features)
        image_features = self.image_projection(image_features)

        # -------- FUSE --------
        fused = torch.cat(
            [text_features, image_features],
            dim=1
        )

        logits = self.fusion_classifier(fused)

        return logits


fusion_model = MuRILCLIPFusion(
    muril_model,
    image_model,
    num_classes=4
).to(device)

trainable = sum(
    p.numel()
    for p in fusion_model.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in fusion_model.parameters()
)

print("Device:", next(fusion_model.parameters()).device)
print(f"Total parameters: {total:,}")
print(f"Trainable parameters: {trainable:,}")

Device: cuda:0
Total parameters: 325,538,308
Trainable parameters: 526,084


In [9]:
batch = next(iter(train_mm_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
pixel_values = batch["pixel_values"].to(device)

with torch.no_grad():
    logits = fusion_model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        pixel_values=pixel_values
    )

print("Logits shape:", logits.shape)
print("Expected:", torch.Size([8, 4]))

Logits shape: torch.Size([8, 4])
Expected: torch.Size([8, 4])


In [10]:
batch = next(iter(train_mm_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
pixel_values = batch["pixel_values"].to(device)

with torch.no_grad():
    logits = fusion_model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        pixel_values=pixel_values
    )

print("Logits shape:", logits.shape)

Logits shape: torch.Size([8, 4])


In [11]:
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)
from tqdm.auto import tqdm

# -------------------------
# CLASS WEIGHTS
# -------------------------
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(labels),
    y=train_split["Category"]
)

mm_class_weights = torch.tensor(
    weights,
    dtype=torch.float32
).to(device)

loss_fn = nn.CrossEntropyLoss(
    weight=mm_class_weights
)

# Only train unfrozen fusion/projection parameters
optimizer = AdamW(
    filter(lambda p: p.requires_grad, fusion_model.parameters()),
    lr=5e-4,
    weight_decay=1e-4
)

EPOCHS = 8

best_mm_f1 = -1
best_mm_state = None
mm_history = []

print("Class weights:", mm_class_weights)
print("Starting multimodal training...")

Class weights: tensor([ 0.8628,  0.9992, 14.2976,  0.5649], device='cuda:0')
Starting multimodal training...


In [12]:
for epoch in range(EPOCHS):

    # =========================
    # TRAIN
    # =========================
    fusion_model.train()

    total_train_loss = 0

    train_bar = tqdm(
        train_mm_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS} - Training"
    )

    for batch in train_bar:

        input_ids = batch["input_ids"].to(
            device, non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            device, non_blocking=True
        )

        pixel_values = batch["pixel_values"].to(
            device, non_blocking=True
        )

        labels_batch = batch["label"].to(
            device, non_blocking=True
        )

        optimizer.zero_grad()

        logits = fusion_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values
        )

        loss = loss_fn(
            logits,
            labels_batch
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            fusion_model.parameters(),
            1.0
        )

        optimizer.step()

        total_train_loss += loss.item()

        train_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    avg_train_loss = (
        total_train_loss / len(train_mm_loader)
    )

    # =========================
    # VALIDATION
    # =========================
    fusion_model.eval()

    total_val_loss = 0
    all_preds = []
    all_true = []

    with torch.no_grad():

        val_bar = tqdm(
            val_mm_loader,
            desc=f"Epoch {epoch+1}/{EPOCHS} - Validation"
        )

        for batch in val_bar:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            pixel_values = batch["pixel_values"].to(device)
            labels_batch = batch["label"].to(device)

            logits = fusion_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values
            )

            loss = loss_fn(
                logits,
                labels_batch
            )

            total_val_loss += loss.item()

            preds = torch.argmax(
                logits,
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_true.extend(
                labels_batch.cpu().numpy()
            )

    avg_val_loss = (
        total_val_loss / len(val_mm_loader)
    )

    # =========================
    # METRICS
    # =========================
    accuracy = accuracy_score(
        all_true,
        all_preds
    )

    precision, recall, macro_f1, _ = (
        precision_recall_fscore_support(
            all_true,
            all_preds,
            average="macro",
            zero_division=0
        )
    )

    weighted_f1 = (
        precision_recall_fscore_support(
            all_true,
            all_preds,
            average="weighted",
            zero_division=0
        )[2]
    )

    cm = confusion_matrix(
        all_true,
        all_preds,
        labels=range(4)
    )

    print("\n" + "=" * 60)
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss:      {avg_train_loss:.4f}")
    print(f"Validation Loss: {avg_val_loss:.4f}")
    print(f"Accuracy:        {accuracy:.4f}")
    print(f"Macro Precision: {precision:.4f}")
    print(f"Macro Recall:    {recall:.4f}")
    print(f"Macro F1:        {macro_f1:.4f}")
    print(f"Weighted F1:     {weighted_f1:.4f}")
    print("Confusion Matrix:")
    print(cm)
    print("=" * 60)

    mm_history.append({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "val_loss": avg_val_loss,
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    })

    # SAVE BEST IN MEMORY
    if macro_f1 > best_mm_f1:

        best_mm_f1 = macro_f1

        best_mm_state = {
            k: v.cpu().clone()
            for k, v in fusion_model.state_dict().items()
        }

        print(
            f"🔥 NEW BEST MULTIMODAL F1: "
            f"{best_mm_f1:.4f}"
        )

print("\n================================")
print("MULTIMODAL TRAINING COMPLETE")
print("Best Multimodal Macro-F1:", best_mm_f1)
print("MuRIL baseline:           0.5336")
print("CLIP baseline:            0.4991")
print("================================")

Epoch 1/8 - Training:   0%|          | 0/301 [00:00<?, ?it/s]

Epoch 1/8 - Validation:   0%|          | 0/75 [00:00<?, ?it/s]


Epoch 1/8
Train Loss:      0.8376
Validation Loss: 1.3833
Accuracy:        0.6317
Macro Precision: 0.4607
Macro Recall:    0.4664
Macro F1:        0.4626
Weighted F1:     0.6317
Confusion Matrix:
[[112  48   0  14]
 [ 34  75   0  41]
 [  1   3   0   6]
 [ 22  52   0 192]]
🔥 NEW BEST MULTIMODAL F1: 0.4626


Epoch 2/8 - Training:   0%|          | 0/301 [00:00<?, ?it/s]

Epoch 2/8 - Validation:   0%|          | 0/75 [00:00<?, ?it/s]

...

Epoch 7/8 - Validation:   0%|          | 0/75 [00:00<?, ?it/s]


Epoch 7/8
Train Loss:      0.4628
Validation Loss: 1.8278
Accuracy:        0.6383
Macro Precision: 0.5497
Macro Recall:    0.5238
Macro F1:        0.5306
Weighted F1:     0.6410
Confusion Matrix:
[[129  35   0  10]
 [ 44  71   0  35]
 [  1   3   2   4]
 [ 25  56   4 181]]


Epoch 8/8 - Training:   0%|          | 0/301 [00:00<?, ?it/s]

Epoch 8/8 - Validation:   0%|          | 0/75 [00:00<?, ?it/s]


Epoch 8/8
Train Loss:      0.4322
Validation Loss: 1.7538
Accuracy:        0.6367
Macro Precision: 0.6223
Macro Recall:    0.5489
Macro F1:        0.5699
Weighted F1:     0.6449
Confusion Matrix:
[[117  47   0  10]
 [ 35  82   0  33]
 [  1   3   3   3]
 [ 21  63   2 180]]

MULTIMODAL TRAINING COMPLETE
Best Multimodal Macro-F1: 0.5743086025857458
MuRIL baseline:           0.5336
CLIP baseline:            0.4991


In [13]:
import os
import torch
import pandas as pd

SAVE_DIR = "/content/drive/MyDrive/IMUSA/results/multimodal"

os.makedirs(SAVE_DIR, exist_ok=True)

# Restore BEST multimodal checkpoint
fusion_model.load_state_dict(best_mm_state)
fusion_model = fusion_model.to(device)

# Save model
torch.save(
    {
        "model_state_dict": fusion_model.state_dict(),
        "label2id": label2id,
        "id2label": id2label,
        "best_macro_f1": best_mm_f1
    },
    f"{SAVE_DIR}/best_model.pt"
)

# Save training history
pd.DataFrame(mm_history).to_csv(
    f"{SAVE_DIR}/training_log.csv",
    index=False
)

print("✅ Multimodal model saved")
print("Location:", SAVE_DIR)
print("Best Macro-F1:", best_mm_f1)

✅ Multimodal model saved
Location: /content/drive/MyDrive/IMUSA/results/multimodal
Best Macro-F1: 0.5743086025857458


In [14]:
import numpy as np
import torch.nn.functional as F

fusion_model.eval()

all_probs = []
all_preds = []
all_true = []
all_ids = []

with torch.no_grad():

    for batch in val_mm_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        pixel_values = batch["pixel_values"].to(device)

        logits = fusion_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values
        )

        probs = F.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(batch["label"].numpy())
        all_ids.extend(batch["id"])

probs_array = np.array(all_probs)

val_mm_results = val_split.copy()

val_mm_results["predicted_label"] = [
    id2label[i] for i in all_preds
]

for i, label in id2label.items():
    val_mm_results[f"prob_{label}"] = probs_array[:, i]

val_mm_results.to_csv(
    f"{SAVE_DIR}/val_predictions.csv",
    index=False
)

print("✅ Validation probabilities saved")
print("Rows:", len(val_mm_results))

✅ Validation probabilities saved
Rows: 600


In [15]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

TEXT_PRED = f"{IMUSA_DIR}/results/text_muril/val_predictions.csv"
MM_PRED = f"{IMUSA_DIR}/results/multimodal/val_predictions.csv"

text_df = pd.read_csv(TEXT_PRED)
mm_df = pd.read_csv(MM_PRED)

print("MuRIL rows:", len(text_df))
print("Multimodal rows:", len(mm_df))

# Ensure predictions correspond to same validation samples
assert text_df["Id"].tolist() == mm_df["Id"].tolist(), \
    "ERROR: Validation IDs/order do not match!"

print("✅ Validation IDs match")

prob_cols = [
    "prob_Motivational",
    "prob_Neutral",
    "prob_Offensive",
    "prob_Sarcasm"
]

P_text = text_df[prob_cols].values
P_mm = mm_df[prob_cols].values

true_labels = mm_df["Category"].map(label2id).values

print("\nTesting ensemble weights...\n")

results = []

# alpha = multimodal weight
for alpha in np.arange(0.0, 1.01, 0.05):

    P_ensemble = (
        alpha * P_mm +
        (1 - alpha) * P_text
    )

    preds = np.argmax(P_ensemble, axis=1)

    macro_f1 = f1_score(
        true_labels,
        preds,
        average="macro"
    )

    accuracy = accuracy_score(
        true_labels,
        preds
    )

    results.append(
        (alpha, macro_f1, accuracy)
    )

    print(
        f"MM={alpha:.2f} | "
        f"MuRIL={1-alpha:.2f} | "
        f"Macro-F1={macro_f1:.4f} | "
        f"Accuracy={accuracy:.4f}"
    )

best = max(results, key=lambda x: x[1])

print("\n==============================")
print("BEST ENSEMBLE")
print(f"Multimodal weight: {best[0]:.2f}")
print(f"MuRIL weight:      {1-best[0]:.2f}")
print(f"Macro-F1:          {best[1]:.4f}")
print(f"Accuracy:          {best[2]:.4f}")
print("==============================")

MuRIL rows: 600
Multimodal rows: 600
✅ Validation IDs match

Testing ensemble weights...

MM=0.00 | MuRIL=1.00 | Macro-F1=0.5336 | Accuracy=0.6267
MM=0.05 | MuRIL=0.95 | Macro-F1=0.5444 | Accuracy=0.6367
MM=0.10 | MuRIL=0.90 | Macro-F1=0.5459 | Accuracy=0.6367
MM=0.15 | MuRIL=0.85 | Macro-F1=0.5428 | Accuracy=0.6317
MM=0.20 | MuRIL=0.80 | Macro-F1=0.5452 | Accuracy=0.6333
MM=0.25 | MuRIL=0.75 | Macro-F1=0.5453 | Accuracy=0.6333
MM=0.30 | MuRIL=0.70 | Macro-F1=0.5420 | Accuracy=0.6283
MM=0.35 | MuRIL=0.65 | Macro-F1=0.5438 | Accuracy=0.6300
MM=0.40 | MuRIL=0.60 | Macro-F1=0.5425 | Accuracy=0.6283
MM=0.45 | MuRIL=0.55 | Macro-F1=0.5425 | Accuracy=0.6283
MM=0.50 | MuRIL=0.50 | Macro-F1=0.5425 | Accuracy=0.6283
MM=0.55 | MuRIL=0.45 | Macro-F1=0.5388 | Accuracy=0.6233
MM=0.60 | MuRIL=0.40 | Macro-F1=0.5390 | Accuracy=0.6233
MM=0.65 | MuRIL=0.35 | Macro-F1=0.5391 | Accuracy=0.6233
MM=0.70 | MuRIL=0.30 | Macro-F1=0.5391 | Accuracy=0.6233
MM=0.75 | MuRIL=0.25 | Macro-F1=0.5400 | Accuracy=0.625

In [17]:
import os

IMUSA_DIR = "/content/drive/MyDrive/IMUSA"

print("Files/folders inside IMUSA:\n")

for item in os.listdir(IMUSA_DIR):
    print(item)

Files/folders inside IMUSA:

split_train.csv
split_val.csv
results
Training_images
Testing_images
Test.csv


In [19]:
IMUSA_DIR = "/content/drive/MyDrive/IMUSA"

TEST_CSV = f"{IMUSA_DIR}/Test.csv"
TEST_IMAGE_DIR = f"{IMUSA_DIR}/Testing_images"

print("Test CSV:", TEST_CSV)
print("Test images:", TEST_IMAGE_DIR)

Test CSV: /content/drive/MyDrive/IMUSA/Test.csv
Test images: /content/drive/MyDrive/IMUSA/Testing_images


In [20]:
import os
import pandas as pd

test_df = pd.read_csv(TEST_CSV)

print("Test shape:", test_df.shape)
print("Test columns:", test_df.columns.tolist())

display(test_df.head())

print("\nImages in test folder:", len(os.listdir(TEST_IMAGE_DIR)))

Test shape: (500, 3)
Test columns: ['Id', 'Category', 'Text']


,Id,Category,Text
0,image_punjabi_1.jpg,NaN,"ਟੌਹਰ ਤਾਂ ਮੇਰੀ ਆ, ਬਾਕੀ ਤਾਂ ਸਾਲਿਆ ਨੇ ਐਵੇਂ ਥਾਂ ਘੇ..."
1,image_punjabi_2.jpg,NaN,“ਚੰਗਾ ਹੋਵੇ ਚੱਲਦੀ ਆ\nਅੱਜ ਬਹੁਤ ਚੁਗਲੀਆਂ ਹੋ ਗਈਆਂ\n...
2,image_punjabi_3.jpg,NaN,ਓਏ ਬੱਸ ਕਰ ਭਰਾਵਾ ਨਾ ਭੋਰਿਆ ਕਰ ਐਨੀ
3,image_punjabi_4.jpg,NaN,“ਰਮਨ – ਬਾਈ ਕੁੜੀਆਂ ਦੇ ਨਜਾਰੇ ਆ\nਇਹਨਾਂ ਨੂੰ ਕੋਈ ਟੈ...
4,image_punjabi_5.jpg,NaN,“ਕਈ ਬੰਦੇ ਬਹੁਤ ਦਿਖਾਵੇ ਵਾਲੇ ਹੁੰਦੇ ਆ\nਇੰਨਾਂ ਦੇ ਪਿ...



Images in test folder: 500


In [21]:
def resolve_test_image_path(image_id):
    image_id = str(image_id).strip()

    # Exact filename
    path = os.path.join(TEST_IMAGE_DIR, image_id)
    if os.path.exists(path):
        return path

    # Try extensions if missing
    for ext in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
        path = os.path.join(TEST_IMAGE_DIR, image_id + ext)
        if os.path.exists(path):
            return path

    return None


missing_test_images = [
    image_id
    for image_id in test_df["Id"]
    if resolve_test_image_path(image_id) is None
]

print("Test rows:", len(test_df))
print("Duplicate IDs:", test_df["Id"].duplicated().sum())
print("Missing Text:", test_df["Text"].isna().sum())
print("Missing images:", len(missing_test_images))

if missing_test_images:
    print("Missing IDs:", missing_test_images[:20])
else:
    print("✅ All 500 test IDs have matching images.")

Test rows: 500
Duplicate IDs: 0
Missing Text: 0
Missing images: 53
Missing IDs: ['image_punjabi_1.jpg', 'image_punjabi_3.jpg', 'image_punjabi_5.jpg', 'image_punjabi_7.jpg', 'image_punjabi_9.jpg', 'image_punjabi_10.jpg', 'image_punjabi_11.jpg', 'image_punjabi_12.jpg', 'image_punjabi_13.jpg', 'image_punjabi_239.jpg', 'image_punjabi_242.jpg', 'image_punjabi_246.jpg', 'image_punjabi_249.jpg', 'image_punjabi_251.jpg', 'image_punjabi_391.jpg', 'image_punjabi_392.jpg', 'image_punjabi_393.jpg', 'image_punjabi_394.jpg', 'image_punjabi_395.jpg', 'image_punjabi_396.jpg']


In [22]:
import os
import re

# Build a numeric-ID -> actual filename map
test_files = os.listdir(TEST_IMAGE_DIR)

numeric_image_map = {}

for filename in test_files:
    match = re.search(r'image_punjabi_(\d+)', filename, re.IGNORECASE)

    if match:
        num = int(match.group(1))

        if num in numeric_image_map:
            print("WARNING duplicate numeric ID:", num,
                  numeric_image_map[num], filename)

        numeric_image_map[num] = filename

print("Files in folder:", len(test_files))
print("Numeric IDs detected:", len(numeric_image_map))

Files in folder: 500
Numeric IDs detected: 500


In [23]:
def resolve_test_image_path(image_id):
    image_id = str(image_id).strip()

    # First try exact filename
    exact = os.path.join(TEST_IMAGE_DIR, image_id)
    if os.path.exists(exact):
        return exact

    # Extract numeric ID from spreadsheet value
    match = re.search(r'image_punjabi_(\d+)', image_id, re.IGNORECASE)

    if match:
        num = int(match.group(1))

        actual_filename = numeric_image_map.get(num)

        if actual_filename is not None:
            return os.path.join(TEST_IMAGE_DIR, actual_filename)

    return None

In [24]:
missing_test_images = [
    image_id
    for image_id in test_df["Id"]
    if resolve_test_image_path(image_id) is None
]

print("Test rows:", len(test_df))
print("Duplicate IDs:", test_df["Id"].duplicated().sum())
print("Missing Text:", test_df["Text"].isna().sum())
print("Missing images:", len(missing_test_images))

if missing_test_images:
    print("Still missing:")
    print(missing_test_images[:30])
else:
    print("✅ All 500 test IDs now resolve to an image.")

Test rows: 500
Duplicate IDs: 0
Missing Text: 0
Missing images: 0
✅ All 500 test IDs now resolve to an image.


In [25]:
class IMUSATestMultimodalDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # TEXT
        text = str(row["Text"])

        text_encoding = tokenizer(
            text,
            max_length=128,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        # IMAGE
        image_id = row["Id"]
        image_path = resolve_test_image_path(image_id)

        if image_path is None:
            raise FileNotFoundError(image_id)

        image = Image.open(image_path).convert("RGB")

        image_encoding = clip_processor(
            images=image,
            return_tensors="pt"
        )

        return {
            "id": image_id,
            "input_ids":
                text_encoding["input_ids"].squeeze(0),
            "attention_mask":
                text_encoding["attention_mask"].squeeze(0),
            "pixel_values":
                image_encoding["pixel_values"].squeeze(0)
        }


test_mm_dataset = IMUSATestMultimodalDataset(test_df)

test_mm_loader = DataLoader(
    test_mm_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Test samples:", len(test_mm_dataset))

batch = next(iter(test_mm_loader))

print("Text:", batch["input_ids"].shape)
print("Images:", batch["pixel_values"].shape)
print("First IDs:", batch["id"][:3])

Test samples: 500
Text: torch.Size([8, 128])
Images: torch.Size([8, 3, 224, 224])
First IDs: ['image_punjabi_1.jpg', 'image_punjabi_2.jpg', 'image_punjabi_3.jpg']


In [26]:
fusion_model.load_state_dict(best_mm_state)
fusion_model = fusion_model.to(device)
fusion_model.eval()

test_mm_probs = []
test_mm_preds = []
test_ids = []

with torch.no_grad():

    for batch in tqdm(
        test_mm_loader,
        desc="Predicting test set"
    ):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        pixel_values = batch["pixel_values"].to(device)

        logits = fusion_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values
        )

        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        test_mm_probs.extend(probs.cpu().numpy())
        test_mm_preds.extend(preds.cpu().numpy())
        test_ids.extend(batch["id"])

test_mm_probs = np.array(test_mm_probs)

print("Predictions:", len(test_mm_preds))
print("Probability shape:", test_mm_probs.shape)

Predicting test set:   0%|          | 0/63 [00:00<?, ?it/s]

Predictions: 500
Probability shape: (500, 4)


In [27]:
from collections import Counter

predicted_labels = [id2label[i] for i in test_mm_preds]

print("Prediction distribution:")
print(Counter(predicted_labels))

print("\nFirst 10 predictions:")
for i in range(10):
    print(test_ids[i], "->", predicted_labels[i])

Prediction distribution:
Counter({'Sarcasm': 395, 'Neutral': 92, 'Motivational': 11, 'Offensive': 2})

First 10 predictions:
image_punjabi_1.jpg -> Sarcasm
image_punjabi_2.jpg -> Sarcasm
image_punjabi_3.jpg -> Neutral
image_punjabi_4.jpg -> Sarcasm
image_punjabi_5.jpg -> Neutral
image_punjabi_6.jpg -> Sarcasm
image_punjabi_7.jpg -> Sarcasm
image_punjabi_8.jpg -> Sarcasm
image_punjabi_9.jpg -> Sarcasm
image_punjabi_10.jpg -> Sarcasm


In [28]:
from collections import Counter

fusion_model.load_state_dict(best_mm_state)
fusion_model.eval()

val_preds_check = []

with torch.no_grad():
    for batch in val_mm_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        pixel_values = batch["pixel_values"].to(device)

        logits = fusion_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values
        )

        preds = torch.argmax(logits, dim=1)
        val_preds_check.extend(preds.cpu().numpy())

val_pred_labels = [id2label[i] for i in val_preds_check]

print("VALIDATION predicted distribution:")
print(Counter(val_pred_labels))

print("\nVALIDATION true distribution:")
print(Counter(val_split["Category"]))

VALIDATION predicted distribution:
Counter({'Sarcasm': 242, 'Neutral': 205, 'Motivational': 150, 'Offensive': 3})

VALIDATION true distribution:
Counter({'Sarcasm': 266, 'Motivational': 174, 'Neutral': 150, 'Offensive': 10})


In [29]:
max_conf = test_mm_probs.max(axis=1)

print("Average confidence:", max_conf.mean())
print("Median confidence:", np.median(max_conf))
print("Min confidence:", max_conf.min())
print("Max confidence:", max_conf.max())

for label in labels:
    idx = [i for i, p in enumerate(predicted_labels) if p == label]

    if idx:
        print(
            label,
            "count =", len(idx),
            "avg confidence =",
            max_conf[idx].mean()
        )

Average confidence: 0.9319994
Median confidence: 0.99430615
Min confidence: 0.50409365
Max confidence: 0.9999249
Motivational count = 11 avg confidence = 0.7954668
Neutral count = 92 avg confidence = 0.8178825
Offensive count = 2 avg confidence = 0.93715274
Sarcasm count = 395 avg confidence = 0.9623546


In [30]:
from collections import Counter

fusion_model.load_state_dict(best_mm_state)
fusion_model = fusion_model.to(device)
fusion_model.eval()

val_preds_check = []

with torch.no_grad():
    for batch in val_mm_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        pixel_values = batch["pixel_values"].to(device)

        logits = fusion_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values
        )

        preds = torch.argmax(logits, dim=1)
        val_preds_check.extend(preds.cpu().numpy())

val_pred_labels = [id2label[i] for i in val_preds_check]

print("VALIDATION TRUE:")
print(Counter(val_split["Category"]))

print("\nVALIDATION PREDICTED:")
print(Counter(val_pred_labels))

print("\nTEST PREDICTED:")
print(Counter(predicted_labels))

VALIDATION TRUE:
Counter({'Sarcasm': 266, 'Motivational': 174, 'Neutral': 150, 'Offensive': 10})

VALIDATION PREDICTED:
Counter({'Sarcasm': 242, 'Neutral': 205, 'Motivational': 150, 'Offensive': 3})

TEST PREDICTED:
Counter({'Sarcasm': 395, 'Neutral': 92, 'Motivational': 11, 'Offensive': 2})


In [31]:
for i in range(20):
    csv_id = test_df.iloc[i]["Id"]
    resolved = resolve_test_image_path(csv_id)

    print(
        f"CSV: {csv_id:25s} -> IMAGE: {os.path.basename(resolved)}"
    )

CSV: image_punjabi_1.jpg       -> IMAGE: image_punjabi_1.jpeg
CSV: image_punjabi_2.jpg       -> IMAGE: image_punjabi_2.jpg
CSV: image_punjabi_3.jpg       -> IMAGE: image_punjabi_3.jpeg
CSV: image_punjabi_4.jpg       -> IMAGE: image_punjabi_4.jpg
CSV: image_punjabi_5.jpg       -> IMAGE: image_punjabi_5.jpeg
CSV: image_punjabi_6.jpg       -> IMAGE: image_punjabi_6.jpg
CSV: image_punjabi_7.jpg       -> IMAGE: image_punjabi_7.jpeg
CSV: image_punjabi_8.jpg       -> IMAGE: image_punjabi_8.jpg
CSV: image_punjabi_9.jpg       -> IMAGE: image_punjabi_9.jpeg
CSV: image_punjabi_10.jpg      -> IMAGE: image_punjabi_10.jpeg
CSV: image_punjabi_11.jpg      -> IMAGE: image_punjabi_11.jpeg
CSV: image_punjabi_12.jpg      -> IMAGE: image_punjabi_12.jpeg
CSV: image_punjabi_13.jpg      -> IMAGE: image_punjabi_13.jpeg
CSV: image_punjabi_14.jpg      -> IMAGE: image_punjabi_14.jpg
CSV: image_punjabi_15.jpg      -> IMAGE: image_punjabi_15.jpg
CSV: image_punjabi_16.jpg      -> IMAGE: image_punjabi_16.jpg
CSV: ima

In [32]:
from torch.utils.data import Dataset, DataLoader

class IMUSATestTextDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        text = str(row["Text"])

        encoding = tokenizer(
            text,
            max_length=128,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "id": row["Id"],
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }


test_text_dataset = IMUSATestTextDataset(test_df)

test_text_loader = DataLoader(
    test_text_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Test text samples:", len(test_text_dataset))

Test text samples: 500


In [33]:
import numpy as np
import torch
from tqdm.auto import tqdm

muril_model.eval()

test_text_probs = []
test_text_preds = []
test_text_ids = []

with torch.no_grad():

    for batch in tqdm(
        test_text_loader,
        desc="MuRIL test prediction"
    ):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = muril_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probs = torch.softmax(outputs.logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        test_text_probs.extend(probs.cpu().numpy())
        test_text_preds.extend(preds.cpu().numpy())
        test_text_ids.extend(batch["id"])

test_text_probs = np.array(test_text_probs)

print("Predictions:", len(test_text_preds))
print("Probability shape:", test_text_probs.shape)

MuRIL test prediction:   0%|          | 0/32 [00:00<?, ?it/s]

Predictions: 500
Probability shape: (500, 4)


In [34]:
from collections import Counter

text_predicted_labels = [
    id2label[i] for i in test_text_preds
]

print("MuRIL TEST distribution:")
print(Counter(text_predicted_labels))

print("\nMultimodal TEST distribution:")
print(Counter(predicted_labels))

MuRIL TEST distribution:
Counter({'Sarcasm': 411, 'Neutral': 63, 'Motivational': 26})

Multimodal TEST distribution:
Counter({'Sarcasm': 395, 'Neutral': 92, 'Motivational': 11, 'Offensive': 2})


In [35]:
assert test_ids == test_text_ids
print(" MuRIL and Multimodal test IDs match exactly.")

 MuRIL and Multimodal test IDs match exactly.


In [36]:
alpha = 0.95

test_ensemble_probs = (
    alpha * test_mm_probs
    + (1 - alpha) * test_text_probs
)

test_ensemble_preds = np.argmax(
    test_ensemble_probs,
    axis=1
)

ensemble_labels = [
    id2label[i]
    for i in test_ensemble_preds
]

print("Ensemble distribution:")
print(Counter(ensemble_labels))

Ensemble distribution:
Counter({'Sarcasm': 395, 'Neutral': 92, 'Motivational': 11, 'Offensive': 2})


In [37]:
import os
import pandas as pd

SUBMISSION_DIR = f"{IMUSA_DIR}/submissions"
os.makedirs(SUBMISSION_DIR, exist_ok=True)

# ---------------------------
# RUN 1: MULTIMODAL
# ---------------------------
run1 = pd.DataFrame({
    "id": test_ids,
    "label": predicted_labels
})

run1.to_csv(
    f"{SUBMISSION_DIR}/run1_multimodal.csv",
    index=False
)

# ---------------------------
# RUN 2: MuRIL
# ---------------------------
run2 = pd.DataFrame({
    "id": test_text_ids,
    "label": text_predicted_labels
})

run2.to_csv(
    f"{SUBMISSION_DIR}/run2_muril.csv",
    index=False
)

print("Run 1:", run1.shape)
print("Run 2:", run2.shape)

print("\nDifferent predictions between Run 1 and Run 2:")
print((run1["label"] != run2["label"]).sum())

print("\nSaved to:")
print(SUBMISSION_DIR)

Run 1: (500, 2)
Run 2: (500, 2)

Different predictions between Run 1 and Run 2:
47

Saved to:
/content/drive/MyDrive/IMUSA/submissions


In [38]:
for name, df in [
    ("run1_multimodal", run1),
    ("run2_muril", run2)
]:
    print("\n", name)

    assert list(df.columns) == ["id", "label"]
    assert len(df) == 500
    assert df["id"].nunique() == 500
    assert df["id"].isna().sum() == 0
    assert df["label"].isna().sum() == 0
    assert set(df["label"]).issubset(set(labels))

    print("✅ VALID")
    print(df["label"].value_counts())


 run1_multimodal
✅ VALID
label
Sarcasm         395
Neutral          92
Motivational     11
Offensive         2
Name: count, dtype: int64

 run2_muril
✅ VALID
label
Sarcasm         411
Neutral          63
Motivational     26
Name: count, dtype: int64


In [39]:
import os
import pandas as pd

FINAL_DIR = f"{IMUSA_DIR}/final_submissions"
os.makedirs(FINAL_DIR, exist_ok=True)

# RUN 1 — 95% Multimodal + 5% MuRIL ensemble
run1 = pd.DataFrame({
    "id": test_ids,
    "label": ensemble_labels
})

# RUN 2 — Pure Multimodal
run2 = pd.DataFrame({
    "id": test_ids,
    "label": predicted_labels
})

# RUN 3 — MuRIL text-only
run3 = pd.DataFrame({
    "id": test_text_ids,
    "label": text_predicted_labels
})

run1.to_csv(
    f"{FINAL_DIR}/run1_ensemble.csv",
    index=False
)

run2.to_csv(
    f"{FINAL_DIR}/run2_multimodal.csv",
    index=False
)

run3.to_csv(
    f"{FINAL_DIR}/run3_muril.csv",
    index=False
)

print("✅ All 3 files created.")

✅ All 3 files created.


In [40]:
allowed_labels = {
    "Sarcasm",
    "Neutral",
    "Offensive",
    "Motivational"
}

runs = {
    "RUN 1 - Ensemble": run1,
    "RUN 2 - Multimodal": run2,
    "RUN 3 - MuRIL": run3
}

for name, df in runs.items():

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("Unique IDs:", df["id"].nunique())
    print("Missing IDs:", df["id"].isna().sum())
    print("Missing labels:", df["label"].isna().sum())
    print("\nDistribution:")
    print(df["label"].value_counts())

    assert df.shape == (500, 2)
    assert list(df.columns) == ["id", "label"]
    assert df["id"].nunique() == 500
    assert df["id"].isna().sum() == 0
    assert df["label"].isna().sum() == 0
    assert set(df["label"]).issubset(allowed_labels)

print("\n🔥 ALL THREE IMUSA SUBMISSIONS ARE VALID 🔥")


RUN 1 - Ensemble
Shape: (500, 2)
Columns: ['id', 'label']
Unique IDs: 500
Missing IDs: 0
Missing labels: 0

Distribution:
label
Sarcasm         395
Neutral          92
Motivational     11
Offensive         2
Name: count, dtype: int64

RUN 2 - Multimodal
Shape: (500, 2)
Columns: ['id', 'label']
Unique IDs: 500
Missing IDs: 0
Missing labels: 0

Distribution:
label
Sarcasm         395
Neutral          92
Motivational     11
Offensive         2
Name: count, dtype: int64

RUN 3 - MuRIL
Shape: (500, 2)
Columns: ['id', 'label']
Unique IDs: 500
Missing IDs: 0
Missing labels: 0

Distribution:
label
Sarcasm         411
Neutral          63
Motivational     26
Name: count, dtype: int64

🔥 ALL THREE IMUSA SUBMISSIONS ARE VALID 🔥


In [41]:
print(
    "Run 1 vs Run 2 differences:",
    (run1["label"] != run2["label"]).sum()
)

print(
    "Run 1 vs Run 3 differences:",
    (run1["label"] != run3["label"]).sum()
)

print(
    "Run 2 vs Run 3 differences:",
    (run2["label"] != run3["label"]).sum()
)

Run 1 vs Run 2 differences: 0
Run 1 vs Run 3 differences: 47
Run 2 vs Run 3 differences: 47
